[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/44_train_pipeline_solution.ipynb)

#  Solution: RL Training Pipeline

Reference solution for a complete RL training loop combining rollout collection, return computation, advantage normalization, PPO-style clipping, and gradient updates.

```text
for epoch:
  for each rollout:
    collect (s, a, r) trajectory
    compute discounted returns G_t
    compute advantages = (G - mean(G)) / (std(G) + eps)
    loss = -(A * action).mean()  (policy gradient direction)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
#  SOLUTION

def train_rl_pipeline(policy, ref_policy, env, optimizer,
                      epochs=10, rollouts_per_epoch=4,
                      gamma=0.99, clip_ratio=0.2):
    """Complete RL training pipeline.
    
    References:
    - Schulman et al. "Proximal Policy Optimization Algorithms"
    - Schulman et al. "High-Dimensional Continuous Control Using
      Generalized Advantage Estimation" (GAE)
    - Standard RL loop structure from stable-baselines3 / RLlib
    """
    losses = []
    
    for epoch in range(epochs):
        epoch_losses = []
        
        for _ in range(rollouts_per_epoch):
            # ---- 1) Collect trajectory ----
            states = []
            actions = []
            rewards = []
            
            state = env.reset()
            done = False
            while not done:
                with torch.no_grad():
                    action = policy(state)
                next_state, reward, done = env.step(action)
                
                states.append(state)
                actions.append(action)
                rewards.append(reward)
                state = next_state
            
            # ---- 2) Compute discounted returns ----
            returns = []
            G = 0.0
            for r in reversed(rewards):
                G = r + gamma * G
                returns.insert(0, G)
            returns = torch.tensor(returns, dtype=torch.float32)
            
            # ---- 3) Normalize advantages ----
            if returns.std() > 0:
                advantages = (returns - returns.mean()) / (returns.std() + 1e-8)
            else:
                advantages = returns - returns.mean()
            
            # ---- 4) Compute policy-gradient loss ----
            actions_t = torch.stack(actions)
            # Policy gradient: -E[A * log_pi(a|s)]  simplified here with direct action
            # Negative because we minimize; advantage pushes actions toward higher returns
            loss = -(advantages.detach() * actions_t).mean()
            
            # ---- 5) Gradient update ----
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_losses.append(loss.item())
        
        losses.append(sum(epoch_losses) / len(epoch_losses))
    
    return losses

In [ ]:
#  Verify

class SimplePolicy(nn.Module):
    def forward(self, x): return x * self.scale.abs()

class SimpleEnv:
    def reset(self): self.i = 0; return torch.tensor(1.0)
    def step(self, a):
        self.i += 1
        return torch.tensor(float(self.i+1)), -float(a), self.i >= 3

p = SimplePolicy()
ref = SimplePolicy()
ref.load_state_dict(p.state_dict())
opt = torch.optim.SGD(p.parameters(), lr=0.01)
losses = train_rl_pipeline(p, ref, SimpleEnv(), opt, epochs=3, rollouts_per_epoch=1)
print("Training losses:", [f"{l:.4f}" for l in losses])

In [ ]:
# Run judge
from torch_judge import check
check("train_pipeline")